# quatsim — Super Heavy RTLS, hot-stage separation to catch

Quaternion-based 6-DOF simulation with predictive RK4 boostback cutoff.
The notebook is kept in sync with `run_mission.py`; do not redefine `fly()`
with an older version below the main mission cell.

The animations use the same wireframe tower and hinged/tracking chopstick
asset as the catch model. Powered phases are explicitly frame-reserved so the
3-engine boostback tail is visible for its real duration instead of flashing by.


In [ ]:
import sys; sys.path.insert(0, '.')
%load_ext autoreload
%autoreload 2

import numpy as np
from quatsim.vehicle import Vehicle
from quatsim.aero import AeroModel, GridFinModel
from quatsim.tower import CatchTower
from quatsim.control import AttitudeController, ControlGains
from quatsim.position import PositionController, PositionGains
from quatsim.phases import FlightSequencer, solve_ignition_altitude, catch_report
from quatsim.mission import (SeparationState, solve_boostback, propagate_flip,
                             build_mission, mission_report)
print('imports ok')

## 1. Vehicle, tower, separation state

`fin_drag_factor = 2.00` is CALIBRATED against Flight 9 (ignition at
~361 m/s). The prior guess of 1.15 gave 712 m/s and corrupted every terminal
result — it saturated the actuators and made the burn-duration basin a knife
edge.

The tower absorbs POSITION error within its envelope (±5 m crossrange,
±3 m downrange — the arms are shorter than V1's, so toward/from the tower is
tighter). It cannot absorb VELOCITY, which is why Flight 6 was waved off when
the tracking link dropped.


In [ ]:
V     = Vehicle(prop_mass=500e3)
aero  = AeroModel()
fins  = GridFinModel()
tower = CatchTower()
TARGET = np.array([0., 0., 105.])

sep = SeparationState(altitude=68e3, downrange=83e3, speed=1500.,
                      flight_path_angle=25., prop_remaining=500e3)

FLIP_DURATION = 4.0
BB_THROTTLE   = 0.87
T33_BURN      = 9.1472
BB_ELEVATION  = 1.0
ALT_BIAS      = 0.0
print(V.summary())


## 2. Solve the burns


In [ ]:
sf   = propagate_flip(V, aero, sep, FLIP_DURATION + 1.0, n_lit=5)
bb0  = solve_boostback(V, aero, sep, start_state=sf,
                       throttle=BB_THROTTLE, bracket=(1., 40.))
land = solve_ignition_altitude(V, aero, bb0['prop_after'], 105., 13, 0.45,
                               v_entry=bb0['arrival_speed'])
print(mission_report(sep, bb0, land))

## 3. Fly it


In [ ]:
seq = FlightSequencer(
    V, aero,
    AttitudeController(V, ControlGains(wn=1.5, zeta=0.8)),
    PositionController(V, PositionGains(wn=0.35, zeta=0.95)),
    fins=fins)

def fly(t33, elev=BB_ELEVATION, dt=0.02, log_every=20):
    bb = dict(bb0)
    bb['duration'] = t33 + 6.0
    segs = build_mission(V, aero, sep, bb, land, TARGET, np.zeros(3),
                         flip_duration=FLIP_DURATION, flip_engines=5,
                         boostback_elevation=np.radians(elev))
    for s in segs:
        if s.name.startswith('boostback'):
            s.throttle = BB_THROTTLE
        if s.name == 'boostback_33':
            s.predictive_boostback = True
            s.boostback_target_altitude = 1200.0
            s.boostback_target_x = -2293.624
            s.boostback_reference_33 = T33_BURN
            s.boostback_tail13 = 3.0
            s.boostback_tail3 = 3.0
            s.duration = 12.0
        if s.name == 'landing':
            s.design_frac = 0.20
            s.k_v = 2.0
            s.v_touch = 0.2
            s.k_lat = 0.9
            s.lat_design_frac = 0.15
            s.max_tilt = np.radians(30.)
            s.v_lat_touch = 0.15
            s.k_pos = 0.02
            s.r_target = TARGET
            s.taper_altitude = 600.0
            s.terminal_taper_altitude = 145.0
            s.two_phase = True
            s.solve_brake = True
            s.brake_margin = 0.40
            s.handover_altitude = 250.0
            s.v_terminal_descent = 8.0
            s.v_lat_handover = 0.5
            s.catch_altitude = 105.0 - ALT_BIAS
    return seq.run(sep.state_vector(), segs, dt=dt, log_every=log_every)

out = fly(T33_BURN)
rep = catch_report(out['state'][-1], TARGET, tower=tower)
print(f"apogee     {out['r'][:,2].max()/1000:8.1f} km")
if seq._boostback_pred is not None:
    print(f"predictive 33-engine cutoff {seq._boostback_pred['remaining_33']:8.4f} s")
print(f"flight     {out['t'][-1]:8.1f} s")
print()
for k, (v, ok) in rep['checks'].items():
    print(f'  {k:22s} {v:9.3f}  {"PASS" if ok else "FAIL"}')
print(f"  {'propellant left':22s} {rep['propellant_remaining_t']:9.2f} t")
print(f"  CAUGHT: {rep['caught']}")


## 4. Engine sequence and attitude


In [ ]:
seg = np.array(out['segment']); prev = None
for i in range(len(out['n_lit'])):
    if out['n_lit'][i] != prev:
        print(f"T+{out['t'][i]:6.1f}s  {out['r'][i,2]/1000:7.2f} km  "
              f"{seg[i]:14s} -> {int(out['n_lit'][i]):2d} engines")
        prev = out['n_lit'][i]

## 5. Figures and animation

The summary GIF is arc-length compressed but explicitly reserves frames for
all powered segments, including the short 3-engine boostback tail. The
realtime MP4 uses true time scaling. Both animations use the updated
wireframe tower with hinged/tracking chopstick arms.


In [ ]:
from quatsim import telemetry as T
from quatsim.realtime import animate_realtime, ensure_ffmpeg

print('ffmpeg:', ensure_ffmpeg())
T.tracking_error(out, path='tracking_error.png')
T.telemetry_panel(out, target=TARGET, path='telemetry_panel.png')
T.animate_mission(out, TARGET, n_frames=220, fps=24,
                  path='mission_summary.gif', hold_seconds=3.0,
                  visual_align_altitude=180.0)
animate_realtime(out, TARGET, path='mission_rt.mp4',
                 speed=6.0, fps=24, hold_seconds=3.0,
                 visual_align_altitude=180.0)


In [ ]:
# Animation sanity check: verify the 3-engine boostback tail is actually
# present in the logged telemetry and therefore has frames available to the GIF.
t = out['t']; n = np.asarray(out['n_lit']); seg = np.asarray(out['segment'])
m = (seg == 'boostback_3') & (n == 3)
if np.any(m):
    print(f"boostback 3-engine interval: {t[m][0]:.2f} -> {t[m][-1]:.2f} s "
          f"({t[m][-1]-t[m][0]:.2f} s logged)")
else:
    print('WARNING: no logged 3-engine boostback samples found')

print('terminal:')
print(f"  tilt       {rep['checks']['tilt_deg'][0]:.3f} deg")
print(f"  velocity   {np.linalg.norm(out['v'][-1]):.3f} m/s")
print(f"  prop       {rep['propellant_remaining_t']:.2f} t")


## 6. THE BACKWARD SOLVE — the next piece of work

The one failing criterion is horizontal speed (3.64 m/s vs a 0.5 limit).
Diagnosed cause: lateral authority is `a_vert · tan(tilt)`, and `a_vert`
collapses from ~54 m/s² at ignition to ~1.0 m/s² as the vertical profile
flattens. All lateral correction has to happen in the first seconds of the
burn; after that the vehicle is effectively ballistic horizontally.

The fix is a two-phase landing burn, solved BACKWARD from what three engines
can do. Arithmetic, at ~340 t:

```
  3 engines :  8.2 MN -> net decel  14.4 m/s^2   (T/W  2.47)
 13 engines : 35.7 MN -> net decel  95.2 m/s^2   (T/W 10.71)
```

**Phase 2 (3 engines, ~18 s per Flight 7).** Near-hover thrust, so
`a_vert ≈ g` and lateral authority is `g·tan(30°) = 5.7 m/s²` for the whole
approach — about 100 m/s of lateral Δv available, against the 3.6 m/s needed.
Descending at ~8 m/s for 17 s covers ~71 m, so **handover at ~180 m**.

**Phase 1 (13 engines).** At FULL throttle 13 engines brake 362 → 8 m/s in
**3.7 s**, not Flight 7's 6–7 s. So the burn must be throttled to ~55%,
giving ~48 m/s²:

```
  brake time     352 / 48          =  7.3 s   <- matches Flight 7 (6-7 s)
  brake distance 362^2 / (2*48)    = 1365 m
  ignition       180 + 1365        = 1545 m   <- matches Flight 9 (1-2 km)
```

All three observations close at once. That is the target profile.

**Implementation gap:** `two_phase=True` exists on the landing Segment, but
the 13-engine tracker arrives at the handover carrying ~40 m/s instead of 8,
so the 18 s precision phase collapses to ~6 s and the result is worse than
the baseline. **The next task is narrow and testable in isolation: make the
13-engine phase deliver 8 m/s at 180 m.** Nothing downstream matters until
that holds.

Two data points would pin it down further: velocity and altitude at the
13→3 downselect, from any livestream.


In [ ]:
# Two-phase profile, currently WORSE than baseline. Left here for the
# next session's work — see the backward solve above.
#
# def fly_two_phase(t33):
#     ... same as fly(), plus on the landing segment:
#     s.two_phase = True
#     s.design_frac = 0.50           # ~55% throttle on 13 engines
#     s.handover_altitude = 180.
#     s.v_terminal_descent = 8.
#     s.downselect_margin = 0.95

## Validation — five independent observations reproduced

| quantity | model | observed | source |
|---|---|---|---|
| ignition speed | 362.8 m/s | 361 m/s | Flight 9 |
| apogee (at 7° elev) | 95.5 km | 95.7 km | Flight 13 |
| 33-engine boostback | 11.01 s | ~11 s | Flight 13 |
| coast duration | 198 s | 204 s | published timeline |
| splashdown downrange | 51.0 km | 51 km | Flight 13 (FITTED) |

Separation downrange was fitted to the last row only. The rest are
predictions. Engine sequence 5/33/13/3/0/13/3 matches flown hardware.

### Rules learned the hard way
1. The shooting solve must use the same `dt` as the evaluation.
2. Every structural change invalidates `T33_BURN` and `BB_ELEVATION`.
3. Identical results across a gain sweep means the gain is not reaching the
   actuator — look upstream, not at the controller.
4. `grep` before patching, and execute the notebook after editing it.
